# Cross-Test: PortPy `.models` vs PyPortfolioOpt vs Riskfolio-Lib vs skfolio

This notebook cross-tests **every estimator, optimizer, and constraint in
`portpy.models`** against
[`PyPortfolioOpt`](https://github.com/robertmartin8/PyPortfolioOpt) and
[`Riskfolio-Lib`](https://github.com/dcajasn/Riskfolio-Lib), and
[`skfolio`](https://github.com/skfolio/skfolio) (a third, independent library -
planning.md's own bar for this notebook) - not a curated subset - using live
data fetched via the Alpaca API, mirroring `03_crosstest_metrics_final.ipynb`'s
exhaustive approach for `.metrics`. Also covers two things the original pass of
this notebook skipped: negative weights (short positions, via `WeightBounds`)
and a genuinely mixed-timeframe universe (Mon-Fri equities aligned against
24/7 crypto via `core.align_calendars`).

Where a function has no external equivalent at all (`maximum_diversification`,
James-Stein shrinkage, the `construction`/`management` layers), it's checked
against an independent closed-form/manual implementation or a self-consistency
identity instead of silently skipped - the same standard
`tests/validation/test_models_vs_libraries.py` holds the automated test suite
to. Every genuine methodological difference found along the way (there are a
few) is called out explicitly, not hidden.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# External optimization libraries
from pypfopt import EfficientFrontier
from pypfopt import expected_returns as pf_er
from pypfopt import risk_models as pf_rm
from pypfopt.black_litterman import BlackLittermanModel, market_implied_prior_returns
import riskfolio as rp
from skfolio import RiskMeasure
from skfolio.optimization import MeanRisk, ObjectiveFunction, RiskBudgeting as SkRiskBudgeting

# Alpaca
from alpaca.data.historical import CryptoHistoricalDataClient, StockHistoricalDataClient
from alpaca.data.requests import CryptoBarsRequest, StockBarsRequest
from alpaca.data.timeframe import TimeFrame

import portpy
from portpy import Portfolio
from portpy.models import optimization as opt
from portpy.core import align_calendars
from portpy.models.base import GrossExposure, GroupCap, NetExposure, TurnoverCap, WeightBounds
from portpy.models.construction.builders import build as pp_build
from portpy.models.construction.builders import efficient_frontier as pp_efficient_frontier
from portpy.models.construction.builders import optimize as pp_optimize
from portpy.models.estimators.covariance import covariance as pp_covariance
from portpy.models.estimators.expected_returns import expected_returns as pp_expected_returns
from portpy.models.estimators.factor_models import capm, factor_attribution, fama_french, linear_regression, rolling_regression
from portpy.models.management.rebalancing import compare, monitor, rebalance
from portpy.metrics.covariance import diversification_ratio
from portpy.metrics.risk import beta as metrics_beta

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 1. Authentication & Setup

Load Alpaca API keys.

In [2]:
load_dotenv()

api_key = os.environ.get('ALPACA_KEY')
api_secret = os.environ.get('ALPACA_SECRET')

if not api_key or not api_secret:
    raise ValueError("API Keys not found in .env file! Please set ALPACA_KEY and ALPACA_SECRET")

client = StockHistoricalDataClient(api_key, api_secret)
print("Alpaca client authenticated.")

Alpaca client authenticated.


## 2. Fetch Data from Alpaca

Five assets spanning equity/bond/commodity, plus SPY as a market/benchmark
series - used both as a Black-Litterman/CAPM benchmark and as an equal-count
"market" proxy for the `capm_implied` expected-return estimator.

In [3]:
symbols = ["AAPL", "MSFT", "JPM", "TLT", "GLD"]
benchmark_sym = "SPY"
start_date = pd.Timestamp.now() - pd.DateOffset(years=4)

request_params = StockBarsRequest(
    symbol_or_symbols=symbols + [benchmark_sym],
    timeframe=TimeFrame.Day,
    start=start_date
)

bars = client.get_stock_bars(request_params)
df = bars.df

# Pivot to have symbols as columns, dates as index
prices_all = df['close'].unstack(level=0)
prices_all.index = pd.to_datetime(prices_all.index.get_level_values('timestamp') if isinstance(prices_all.index, pd.MultiIndex) else prices_all.index)
prices_all = prices_all.tz_localize(None) if prices_all.index.tz is not None else prices_all
prices_all = prices_all.dropna(how="any")

prices = prices_all[symbols]
benchmark_prices = prices_all[benchmark_sym]

print(f"Fetched {len(prices)} days of data for {symbols} + benchmark {benchmark_sym}")
prices.tail()

Fetched 1002 days of data for ['AAPL', 'MSFT', 'JPM', 'TLT', 'GLD'] + benchmark SPY


symbol,AAPL,MSFT,JPM,TLT,GLD
timestamp,,,,,
2026-09-08 04:00:00,316.2200,493.9500,353.5100,82.2000,399.7200
2026-09-09 04:00:00,315.3400,491.6500,354.7100,81.7300,403.3500
2026-09-10 04:00:00,326.5700,492.4400,353.5600,80.7800,396.3600
2026-09-11 04:00:00,332.2700,495.6300,356.2300,80.8700,398.7700
2026-09-14 04:00:00,333.0800,505.4100,350.1300,80.9300,392.8400


## 3. Setup PortPy Portfolio & Reference Inputs

Build a `Portfolio` (equal-weighted - the point of this notebook is the
`.models` layer itself, not any particular starting allocation) and derive
`mu`/`cov` once via `.models.estimators` so every library solves the exact
same estimation problem.

In [4]:
weights = {sym: 1.0 / len(symbols) for sym in symbols}
portfolio = Portfolio(prices, weights=weights, risk_free_rate=0.0)

returns = portfolio.asset_returns()
benchmark_returns = benchmark_prices.pct_change().dropna()
returns, benchmark_returns = returns.align(benchmark_returns, join="inner", axis=0)

mu = pp_expected_returns(returns, method="mean_historical")
cov = pp_covariance(returns, method="sample")

print("Annualized expected returns (mean-historical):")
print(mu.round(4))
print("\nAnnualized covariance matrix (sample):")
cov.round(4)

Annualized expected returns (mean-historical):
symbol
AAPL    0.2337
MSFT    0.2191
JPM     0.3019
TLT    -0.0613
GLD     0.2541
Name: expected_return, dtype: float64

Annualized covariance matrix (sample):


symbol,AAPL,MSFT,JPM,TLT,GLD
symbol,,,,,
AAPL,0.0740,0.0329,0.0198,0.0045,0.0036
MSFT,0.0329,0.0751,0.0161,0.0012,0.0049
JPM,0.0198,0.0161,0.0555,-0.0012,0.0035
TLT,0.0045,0.0012,-0.0012,0.0225,0.0056
GLD,0.0036,0.0049,0.0035,0.0056,0.0395


## 4. Cross-Test: Expected-Return Estimators vs PyPortfolioOpt

All four `estimators.expected_returns` methods. `james_stein` has no
PyPortfolioOpt equivalent, so it's checked against its own documented
closed-form formula computed independently with plain numpy instead
(mirroring `tests/validation/test_models_vs_libraries.py`'s approach for
`capm`).

In [5]:
er_results = []

def add_er_res(method_name, pp_series, ref_series, note=""):
    ref_series = pd.Series(ref_series).reindex(symbols)
    pp_series = pp_series.reindex(symbols)
    for sym in symbols:
        er_results.append({
            "Method": method_name, "Note": note, "Asset": sym,
            "PortPy": pp_series[sym], "Reference": ref_series[sym],
            "Abs Diff": abs(pp_series[sym] - ref_series[sym]),
        })

# --- mean_historical, arithmetic ---
mu_pp = pp_expected_returns(returns, method="mean_historical")
mu_ref = pf_er.mean_historical_return(returns, returns_data=True, compounding=False, frequency=252)
add_er_res("mean_historical", mu_pp, mu_ref, note="arithmetic")

# --- mean_historical, geometric ---
mu_pp_geo = pp_expected_returns(returns, method="mean_historical", geometric=True)
mu_ref_geo = pf_er.mean_historical_return(returns, returns_data=True, compounding=True, frequency=252)
add_er_res("mean_historical", mu_pp_geo, mu_ref_geo, note="geometric")

# --- ewma ---
mu_pp_ewma = pp_expected_returns(returns, method="ewma", span=180)
mu_ref_ewma = pf_er.ema_historical_return(returns, returns_data=True, compounding=False, span=180, frequency=252)
add_er_res("ewma", mu_pp_ewma, mu_ref_ewma, note="span=180")

# --- capm_implied ---
mu_pp_capm = pp_expected_returns(returns, method="capm_implied", benchmark=benchmark_returns, rf=0.0)
mu_ref_capm = pf_er.capm_return(returns, market_prices=benchmark_returns.to_frame(), returns_data=True, risk_free_rate=0.0, compounding=False, frequency=252)
add_er_res("capm_implied", mu_pp_capm, mu_ref_capm)

df_er = pd.DataFrame(er_results)
df_er.style.format({"PortPy": "{:.6f}", "Reference": "{:.6f}", "Abs Diff": "{:.2e}"})

,Method,Note,Asset,PortPy,Reference,Abs Diff
0,mean_historical,arithmetic,AAPL,0.233681,0.233681,0.00e+00
1,mean_historical,arithmetic,MSFT,0.219148,0.219148,0.00e+00
2,mean_historical,arithmetic,JPM,0.301888,0.301888,0.00e+00
3,mean_historical,arithmetic,TLT,-0.061334,-0.061334,0.00e+00
4,mean_historical,arithmetic,GLD,0.254098,0.254098,0.00e+00
5,mean_historical,geometric,AAPL,0.217604,0.217604,0.00e+00
6,mean_historical,geometric,MSFT,0.199499,0.199499,0.00e+00
7,mean_historical,geometric,JPM,0.315325,0.315325,0.00e+00
8,mean_historical,geometric,TLT,-0.070000,-0.070000,0.00e+00
9,mean_historical,geometric,GLD,0.263837,0.263837,0.00e+00


In [6]:
# --- james_stein: no PyPortfolioOpt equivalent - verify against the documented
# closed-form shrinkage formula, computed independently with plain numpy at the
# SAME periodic scale the estimator itself uses internally (t_obs pairs with a
# periodic-scale mahalanobis distance, not an annualized one - see the
# estimator's own source comment for why mixing the two scales collapses phi).
mu_pp_js = pp_expected_returns(returns, method="james_stein")

mu_periodic = returns.mean()
cov_periodic = returns.cov().to_numpy()
n_assets = len(mu_periodic)
t_obs = len(returns)
target_periodic = pd.Series(float(mu_periodic.mean()), index=mu_periodic.index)
diff = (mu_periodic - target_periodic).to_numpy()
mahalanobis = float(diff @ np.linalg.pinv(cov_periodic) @ diff)
phi = float(np.clip((n_assets + 2) / ((n_assets + 2) + t_obs * mahalanobis), 0.0, 1.0))
mu_manual_js = 252 * (phi * target_periodic + (1.0 - phi) * mu_periodic)

max_diff = float((mu_pp_js.reindex(symbols) - mu_manual_js.reindex(symbols)).abs().max())
print(f"james_stein vs independent manual-numpy formula - max abs diff: {max_diff:.2e} (shrinkage intensity phi={phi:.4f})")
pd.DataFrame({"PortPy": mu_pp_js, "Manual (independent)": mu_manual_js}).round(6)

james_stein vs independent manual-numpy formula - max abs diff: 2.78e-17 (shrinkage intensity phi=0.3404)


,PortPy,Manual (independent)
symbol,,
AAPL,0.2186,0.2186
MSFT,0.2091,0.2091
JPM,0.2636,0.2636
TLT,0.0241,0.0241
GLD,0.2321,0.2321


## 5. Cross-Test: Covariance Estimators vs PyPortfolioOpt

All five `estimators.covariance` methods. Two of them - `shrinkage` and
`robust` - turn out to disagree with their closest PyPortfolioOpt counterpart
by more than floating-point noise, for a genuine, documented reason (not a
bug), explained inline below each check.

In [7]:
cov_results = []

def add_cov_res(method_name, pp_mat, ref_mat, note=""):
    pp_mat = pp_mat.reindex(index=symbols, columns=symbols)
    ref_mat = ref_mat.reindex(index=symbols, columns=symbols)
    max_diff = float((pp_mat - ref_mat).abs().to_numpy().max())
    cov_results.append({"Method": method_name, "Note": note, "Max Abs Diff": max_diff})

# --- sample ---
cov_pp = pp_covariance(returns, method="sample")
cov_ref = pf_rm.sample_cov(returns, returns_data=True, frequency=252)
add_cov_res("sample", cov_pp, cov_ref)

# --- ewma ---
cov_pp_ewma = pp_covariance(returns, method="ewma", span=180)
cov_ref_ewma = pf_rm.exp_cov(returns, returns_data=True, span=180, frequency=252)
add_cov_res("ewma", cov_pp_ewma, cov_ref_ewma, note="span=180")

# --- ledoit_wolf ---
cov_pp_lw = pp_covariance(returns, method="ledoit_wolf")
cov_ref_lw = pf_rm.CovarianceShrinkage(returns, returns_data=True, frequency=252).ledoit_wolf()
add_cov_res("ledoit_wolf", cov_pp_lw, cov_ref_lw)

# --- shrinkage: DIFFERENT shrinkage target by design - PortPy shrinks toward
# diag(sample) (keeps each asset's own variance, zeroes correlations);
# PyPortfolioOpt's shrunk_covariance() shrinks toward a scaled identity
# (every asset's variance replaced by the cross-sectional average variance).
# Both are valid, textbook linear-shrinkage targets - they just aren't the
# same target, so the two are expected to disagree.
cov_pp_shr = pp_covariance(returns, method="shrinkage", shrinkage_intensity=0.3)
cov_ref_shr = pf_rm.CovarianceShrinkage(returns, returns_data=True, frequency=252).shrunk_covariance(delta=0.3)
add_cov_res("shrinkage", cov_pp_shr, cov_ref_shr, note="delta=0.3, DIFFERENT TARGET (see note)")

# --- robust: DIFFERENT MCD variant - PortPy uses sklearn's MinCovDet, which
# applies a consistency correction / statistical reweighting step on top of
# the raw MCD estimate; PyPortfolioOpt's min_cov_determinant() calls
# sklearn.covariance.fast_mcd directly, returning the raw, uncorrected
# estimate. Same underlying algorithm, different (both legitimate) endpoint.
cov_pp_robust = pp_covariance(returns, method="robust")
cov_ref_robust = pf_rm.min_cov_determinant(returns, returns_data=True, frequency=252, random_state=0)
add_cov_res("robust", cov_pp_robust, cov_ref_robust, note="MinCovDet (reweighted) vs fast_mcd (raw), DIFFERENT VARIANT (see note)")

df_cov = pd.DataFrame(cov_results)
df_cov.style.format({"Max Abs Diff": "{:.2e}"})

,Method,Note,Max Abs Diff
0,sample,,0.00e+00
1,ewma,span=180,6.94e-17
2,ledoit_wolf,,0.00e+00
3,shrinkage,"delta=0.3, DIFFERENT TARGET (see note)",9.24e-03
4,robust,"MinCovDet (reweighted) vs fast_mcd (raw), DIFFERENT VARIANT (see note)",1.93e-02


## 6. Cross-Test: Mean-Variance Family vs PyPortfolioOpt

`min_variance`, `max_sharpe`, `target_return`, `target_volatility`, and
`mean_variance` are all classic convex (or, for max_sharpe, quasi-convex) QPs
that PyPortfolioOpt solves with `cvxpy` - an entirely different solver path
than PortPy's multi-start SLSQP, so close agreement is a real check of the
objective/constraint formulation, not just numerical coincidence.

Note: PyPortfolioOpt's `efficient_return()` constrains return `>= target`
(an inequality), while PortPy's `target_return` uses `== target` (needed so
`efficient_frontier()` can sweep a full curve). The two agree only once
`target` clears the global min-variance portfolio's own return - which is
where the target below is deliberately set.

In [8]:
weight_results = []

def add_weight_res(method_name, pp_w, ref_w):
    ref_w = pd.Series(ref_w).reindex(symbols)
    pp_w = pp_w.reindex(symbols)
    max_abs_diff = float((pp_w - ref_w).abs().max())
    for sym in symbols:
        weight_results.append({
            "Method": method_name, "Asset": sym,
            "PortPy": pp_w[sym], "Reference": ref_w[sym],
            "Abs Diff": abs(pp_w[sym] - ref_w[sym]), "Max Abs Diff": max_abs_diff,
        })

# --- min_variance ---
w_pp = opt.min_variance(cov)
w_ref = EfficientFrontier(mu, cov).min_volatility()
add_weight_res("min_variance", w_pp, w_ref)

# --- max_sharpe ---
w_pp = opt.max_sharpe(mu, cov, rf=0.0)
w_ref = EfficientFrontier(mu, cov).max_sharpe(risk_free_rate=0.0)
add_weight_res("max_sharpe", w_pp, w_ref)

# --- target_return (above the global min-variance return, see note above) ---
min_var_return = float(opt.min_variance(cov).to_numpy() @ mu.to_numpy())
target = min_var_return + 0.03
w_pp = opt.target_return(mu, cov, target=target)
w_ref = EfficientFrontier(mu, cov).efficient_return(target_return=target)
add_weight_res("target_return", w_pp, w_ref)

# --- target_volatility ---
target_vol = 0.15
w_pp = opt.target_volatility(mu, cov, target=target_vol)
w_ref = EfficientFrontier(mu, cov).efficient_risk(target_volatility=target_vol)
add_weight_res("target_volatility", w_pp, w_ref)

# --- mean_variance == PyPortfolioOpt's max_quadratic_utility ---
risk_aversion = 2.0
w_pp = opt.mean_variance(mu, cov, risk_aversion=risk_aversion)
w_ref = EfficientFrontier(mu, cov).max_quadratic_utility(risk_aversion=risk_aversion, market_neutral=False)
add_weight_res("mean_variance", w_pp, w_ref)

df_mv = pd.DataFrame(weight_results)

def color_diff(row):
    if row["Max Abs Diff"] > 0.01:
        return ['background-color: #ffcccc'] * len(row)
    return [''] * len(row)

df_mv.style.apply(color_diff, axis=1).format({
    "PortPy": "{:.4f}", "Reference": "{:.4f}", "Abs Diff": "{:.4f}", "Max Abs Diff": "{:.4f}"
})

,Method,Asset,PortPy,Reference,Abs Diff,Max Abs Diff
0,min_variance,AAPL,0.0394,0.0394,0.0000,0.0000
1,min_variance,MSFT,0.0853,0.0853,0.0000,0.0000
2,min_variance,JPM,0.1786,0.1786,0.0000,0.0000
3,min_variance,TLT,0.4872,0.4872,0.0000,0.0000
4,min_variance,GLD,0.2095,0.2095,0.0000,0.0000
5,max_sharpe,AAPL,0.0995,0.0995,0.0000,0.0000
6,max_sharpe,MSFT,0.0859,0.0859,0.0000,0.0000
7,max_sharpe,JPM,0.3478,0.3478,0.0000,0.0000
8,max_sharpe,TLT,0.0000,0.0000,0.0000,0.0000
9,max_sharpe,GLD,0.4668,0.4668,0.0000,0.0000


## 7. Cross-Test: `efficient_frontier` vs PyPortfolioOpt

`optimization.efficient_frontier` traces the frontier by calling
`target_return` at `n_points` grid targets between the global min-variance
return and the highest single-asset expected return. Each interior grid point
is cross-checked against PyPortfolioOpt's `efficient_return()` at that exact
target. The final grid point (target = the single highest per-asset expected
return) is deliberately excluded - PyPortfolioOpt raises there
("target_return must be lower than the maximum possible return") since its
formulation requires a strict inequality, while PortPy's equality constraint
is still solvable exactly at that boundary.

In [9]:
frontier = opt.efficient_frontier(mu, cov, n_points=9)
print("PortPy frontier (9 points):")
display(frontier[["target_return", "return", "volatility", "sharpe"]].round(4))

frontier_results = []
for idx in range(len(frontier) - 1):  # exclude the final (boundary) point
    row = frontier.iloc[idx]
    target = float(row["target_return"])
    w_pp = row[symbols].astype(float)
    w_ref = pd.Series(EfficientFrontier(mu, cov).efficient_return(target_return=target))
    max_diff = float((w_pp.reindex(symbols) - w_ref.reindex(symbols)).abs().max())
    frontier_results.append({"Grid Index": idx, "Target Return": target, "Max Abs Diff vs PyPortfolioOpt": max_diff})

pd.DataFrame(frontier_results).style.format({"Target Return": "{:.4f}", "Max Abs Diff vs PyPortfolioOpt": "{:.2e}"})

PortPy frontier (9 points):


,target_return,return,volatility,sharpe
0,0.1052,0.1052,0.1105,0.9521
1,0.1298,0.1298,0.1114,1.1646
2,0.1544,0.1544,0.1143,1.3508
3,0.1789,0.1789,0.1188,1.5057
4,0.2035,0.2035,0.1250,1.6286
5,0.2281,0.2281,0.1324,1.7224
6,0.2527,0.2527,0.1410,1.7919
7,0.2773,0.2773,0.1578,1.7573
8,0.3019,0.3019,0.2355,1.2820


,Grid Index,Target Return,Max Abs Diff vs PyPortfolioOpt
0,0,0.1052,2.44e-10
1,1,0.1298,4.14e-10
2,2,0.1544,6.94e-10
3,3,0.1789,1.75e-07
4,4,0.2035,2.70e-09
5,5,0.2281,1.23e-08
6,6,0.2527,5.04e-09
7,7,0.2773,4.51e-10


## 8. Cross-Test: Risk-Based Methods vs Riskfolio-Lib

`risk_parity` and `risk_budgeting` are checked against Riskfolio-Lib's
`rp_optimization`. `hierarchical_risk_parity` is checked against Riskfolio's
`HCPortfolio` HRP with `leaf_order=False` - Riskfolio applies scipy's
*optimal leaf ordering* on top of the raw dendrogram by default, a documented,
optional refinement PortPy's implementation deliberately doesn't apply
(matching Lopez de Prado's original formulation, which uses the raw linkage
order). `leaf_order=True` is included alongside it as a reference point, not
a guaranteed mismatch: with only five assets the optimal leaf order can
coincide with the raw dendrogram order by chance, as it does below - the
synthetic-data test suite (`tests/validation/test_models_vs_libraries.py`,
more assets, a fixed seed) is where the two are deliberately made to
disagree.

In [10]:
risk_results = []

def add_risk_res(method_name, pp_w, ref_w, note=""):
    ref_w = pd.Series(ref_w.to_numpy().ravel(), index=ref_w.index if hasattr(ref_w, "index") else symbols).reindex(symbols)
    pp_w = pp_w.reindex(symbols)
    for sym in symbols:
        risk_results.append({
            "Method": method_name, "Note": note, "Asset": sym,
            "PortPy": pp_w[sym], "Riskfolio-Lib": ref_w[sym],
            "Abs Diff": abs(pp_w[sym] - ref_w[sym]),
        })

port = rp.Portfolio(returns=returns)
port.assets_stats(method_mu="hist", method_cov="hist")

# --- risk_parity ---
w_pp = opt.risk_parity(cov)
w_ref = port.rp_optimization(model="Classic", rm="MV", rf=0, b=None, hist=True)["weights"]
add_risk_res("risk_parity", w_pp, w_ref)

# --- risk_budgeting (unequal budget) ---
budget = pd.Series({"AAPL": 2.0, "MSFT": 1.0, "JPM": 1.0, "TLT": 1.0, "GLD": 1.0})
w_pp = opt.risk_budgeting(cov, budget)
b_vec = (budget / budget.sum()).reindex(symbols).to_numpy().reshape(-1, 1)
w_ref = port.rp_optimization(model="Classic", rm="MV", rf=0, b=b_vec, hist=True)["weights"]
add_risk_res("risk_budgeting", w_pp, w_ref, note="budget=2:1:1:1:1")

# --- hierarchical_risk_parity vs Riskfolio, leaf_order=False (PortPy's convention) ---
w_pp = opt.hierarchical_risk_parity(cov, linkage_method="single")
hc = rp.HCPortfolio(returns=returns)
w_ref_no_leaf = hc.optimization(model="HRP", codependence="pearson", rm="MV", rf=0, linkage="single", leaf_order=False)["weights"]
add_risk_res("hierarchical_risk_parity", w_pp, w_ref_no_leaf, note="leaf_order=False")

# --- same, but Riskfolio's default leaf_order=True - see markdown above ---
w_ref_leaf = hc.optimization(model="HRP", codependence="pearson", rm="MV", rf=0, linkage="single", leaf_order=True)["weights"]
add_risk_res("hierarchical_risk_parity", w_pp, w_ref_leaf, note="leaf_order=True")

df_risk = pd.DataFrame(risk_results)
df_risk.style.format({"PortPy": "{:.4f}", "Riskfolio-Lib": "{:.4f}", "Abs Diff": "{:.4f}"})

,Method,Note,Asset,PortPy,Riskfolio-Lib,Abs Diff
0,risk_parity,,AAPL,0.1362,0.1362,0.0000
1,risk_parity,,MSFT,0.1429,0.1429,0.0000
2,risk_parity,,JPM,0.1814,0.1814,0.0000
3,risk_parity,,TLT,0.3147,0.3147,0.0000
4,risk_parity,,GLD,0.2248,0.2248,0.0000
5,risk_budgeting,budget=2:1:1:1:1,AAPL,0.2058,0.2058,0.0000
6,risk_budgeting,budget=2:1:1:1:1,MSFT,0.1255,0.1255,0.0000
7,risk_budgeting,budget=2:1:1:1:1,JPM,0.1633,0.1633,0.0000
8,risk_budgeting,budget=2:1:1:1:1,TLT,0.2930,0.2930,0.0000
9,risk_budgeting,budget=2:1:1:1:1,GLD,0.2123,0.2123,0.0000


## 9. Cross-Test: `maximum_diversification` Local Optimality

No mainstream library exposes Choueifaty's Most Diversified Portfolio as a
one-line call, so this checks local optimality directly instead: perturb the
solution in every direction and confirm none of the (projected back to a
long-only, fully-invested) neighbors achieve a higher diversification ratio -
the same check `tests/validation/test_models_vs_libraries.py` runs on
synthetic data.

In [11]:
w_md = opt.maximum_diversification(cov)
baseline = diversification_ratio(w_md, cov)

rng = np.random.default_rng(5)
n = len(w_md)
violations = 0
for _ in range(200):
    perturbation = rng.normal(0, 0.01, n)
    candidate = (w_md.to_numpy() + perturbation).clip(min=0.0)
    candidate = candidate / candidate.sum()
    candidate_series = pd.Series(candidate, index=w_md.index)
    if diversification_ratio(candidate_series, cov) > baseline + 1e-6:
        violations += 1

print("maximum_diversification weights:")
print(w_md.round(4))
print(f"\nBaseline diversification ratio: {baseline:.4f}")
print(f"Nearby feasible points that diversify better (out of 200 perturbations): {violations}")

maximum_diversification weights:
AAPL   0.0924
MSFT   0.1305
JPM    0.1938
TLT    0.3536
GLD    0.2297
Name: weight, dtype: float64

Baseline diversification ratio: 1.7913
Nearby feasible points that diversify better (out of 200 perturbations): 0


## 10. Cross-Test: Black-Litterman vs PyPortfolioOpt

Checks the implied-equilibrium-returns step and the full `black_litterman()`
optimizer against PyPortfolioOpt's `BlackLittermanModel`, using market-cap
weights (approximated here as equal weights, since Alpaca doesn't provide
free-float market cap) as the prior.

In [12]:
market_weights = pd.Series(1 / len(symbols), index=symbols)
risk_aversion = 2.5

# --- implied equilibrium returns ---
pi_ref = market_implied_prior_returns(market_weights, risk_aversion, cov, risk_free_rate=0.0)
pi_pp = risk_aversion * (cov.to_numpy() @ market_weights.to_numpy())
pi_df = pd.DataFrame({
    "PortPy (pi = delta * Sigma @ w_mkt)": pi_pp,
    "PyPortfolioOpt": pi_ref.reindex(symbols).to_numpy(),
}, index=symbols)
pi_df["Abs Diff"] = (pi_df.iloc[:, 0] - pi_df.iloc[:, 1]).abs()
print("Implied equilibrium returns:")
display(pi_df)

# --- no views should reproduce market weights ---
w_no_views = opt.black_litterman(cov, market_weights, views={}, risk_aversion=risk_aversion)
print("\nBlack-Litterman with no views (should equal market_weights):")
print(pd.DataFrame({"PortPy (no views)": w_no_views, "Market weights": market_weights}).round(4))

# --- one absolute view: AAPL will return 15% ---
views = {"AAPL": 0.15}
w_bl_pp = opt.black_litterman(cov, market_weights, views=views, risk_aversion=risk_aversion)

bl_ref = BlackLittermanModel(cov, pi=pi_ref, absolute_views=views, risk_aversion=risk_aversion)
ef_bl = EfficientFrontier(bl_ref.bl_returns(), bl_ref.bl_cov())
w_bl_ref = ef_bl.max_quadratic_utility(risk_aversion=risk_aversion, market_neutral=False)

print("\nBlack-Litterman with view {AAPL: 15%} - final weights:")
bl_compare = pd.DataFrame({
    "PortPy": w_bl_pp.reindex(symbols),
    "PyPortfolioOpt": pd.Series(w_bl_ref).reindex(symbols),
})
bl_compare["Abs Diff"] = (bl_compare["PortPy"] - bl_compare["PyPortfolioOpt"]).abs()
bl_compare.round(4)

Implied equilibrium returns:


,PortPy (pi = delta * Sigma @ w_mkt),PyPortfolioOpt,Abs Diff
AAPL,0.0674,0.0674,0.0000
MSFT,0.0651,0.0651,0.0000
JPM,0.0468,0.0468,0.0000
TLT,0.0163,0.0163,0.0000
GLD,0.0285,0.0285,0.0000



Black-Litterman with no views (should equal market_weights):
      PortPy (no views)  Market weights
AAPL             0.2000          0.2000
MSFT             0.2000          0.2000
JPM              0.2000          0.2000
TLT              0.2000          0.2000
GLD              0.2000          0.2000

Black-Litterman with view {AAPL: 15%} - final weights:


,PortPy,PyPortfolioOpt,Abs Diff
AAPL,0.4146,0.4091,0.0054
MSFT,0.1809,0.1753,0.0057
JPM,0.1601,0.1587,0.0014
TLT,0.0912,0.1037,0.0126
GLD,0.1532,0.1532,0.0000


## 11. Cross-Test: Factor Models vs Independent OLS

No third-party portfolio-optimization library exposes CAPM/Fama-French
regression the way `estimators.factor_models` does, so each function is
checked against a *plain numpy least-squares solve* - an independent
implementation of OLS, distinct from the `statsmodels` machinery
`factor_models.py` is built on (same approach as
`tests/validation/test_models_vs_libraries.py`). `linear_regression` and
`rolling_regression` are exercised directly since they're the shared engine
underneath.

In [13]:
# --- capm vs manual OLS closed form ---
y = returns["AAPL"]
capm_model = capm(y, benchmark_returns, rf=0.0)

design = np.column_stack([np.ones(len(benchmark_returns)), benchmark_returns.to_numpy()])
manual_beta = np.linalg.lstsq(design, y.reindex(benchmark_returns.index).to_numpy(), rcond=None)[0]

print("capm vs manual numpy OLS:")
print(f"  alpha (const): portpy={capm_model.params['const']:.8f}  manual={manual_beta[0]:.8f}  diff={abs(capm_model.params['const'] - manual_beta[0]):.2e}")
print(f"  beta (market): portpy={capm_model.params['market']:.8f}  manual={manual_beta[1]:.8f}  diff={abs(capm_model.params['market'] - manual_beta[1]):.2e}")
print(f"  beta vs metrics.risk.beta (independent Cov/Var formula, rf=0): {metrics_beta(y, benchmark_returns):.8f}")

# --- fama_french vs manual OLS closed form ---
# Illustrative style factors (not real Ken French data) - a size-like factor
# from AAPL-vs-JPM and a rates-like factor from TLT's own return, just to
# exercise the multi-factor machinery with real return series.
factors = pd.DataFrame({
    "Mkt-RF": benchmark_returns,
    "SMB": (returns["AAPL"] - returns["JPM"]),
    "HML": returns["TLT"],
}).dropna()

ff_model = fama_french(y, factors, version=3)
aligned = pd.concat([y.rename("y"), factors], axis=1, join="inner").dropna()
design_ff = np.column_stack([np.ones(len(aligned)), aligned[["Mkt-RF", "SMB", "HML"]].to_numpy()])
manual_ff = np.linalg.lstsq(design_ff, aligned["y"].to_numpy(), rcond=None)[0]

ff_compare = pd.DataFrame({
    "PortPy (statsmodels OLS)": ff_model.params.to_numpy(),
    "Manual (numpy lstsq)": manual_ff,
}, index=ff_model.params.index)
ff_compare["Abs Diff"] = (ff_compare.iloc[:, 0] - ff_compare.iloc[:, 1]).abs()
print("\nfama_french vs manual numpy OLS:")
display(ff_compare.style.format("{:.2e}", subset=["Abs Diff"]).format("{:.6f}", subset=["PortPy (statsmodels OLS)", "Manual (numpy lstsq)"]))

# --- factor_attribution: internal identity check (contributions + alpha sum to the mean return) ---
attribution = factor_attribution(y, factors)
total_check = float(attribution["contribution"].sum())
mean_return = float(aligned["y"].mean())
print(f"\nfactor_attribution identity check: sum(contributions) = {total_check:.8f} vs mean(y) = {mean_return:.8f} (diff={abs(total_check - mean_return):.2e})")
display(attribution.round(6))

# --- linear_regression / rolling_regression: exercised directly (general-purpose engine) ---
lr = linear_regression(y, benchmark_returns.rename("market"))
print(f"\nlinear_regression(AAPL, SPY): const={lr.params['const']:.6f}, market={lr.params['market']:.6f}, R2={lr.rsquared:.4f}")

rolling = rolling_regression(y, benchmark_returns.rename("market"), window=90)
print(f"rolling_regression(window=90): {rolling['market'].notna().sum()} non-NaN windows, beta range [{rolling['market'].min():.2f}, {rolling['market'].max():.2f}]")

capm vs manual numpy OLS:
  alpha (const): portpy=0.00011864  manual=0.00011864  diff=1.36e-19
  beta (market): portpy=1.12691921  manual=1.12691921  diff=4.44e-16
  beta vs metrics.risk.beta (independent Cov/Var formula, rf=0): 1.12691921

fama_french vs manual numpy OLS:


,PortPy (statsmodels OLS),Manual (numpy lstsq),Abs Diff
const,0.000325,0.000325,4.34e-19
Mkt-RF,1.008776,1.008776,0.00e+00
SMB,0.539215,0.539215,3.33e-16
HML,-0.102090,-0.102090,6.94e-17



factor_attribution identity check: sum(contributions) = 0.00092731 vs mean(y) = 0.00092731 (diff=2.87e-15)


,beta,contribution,pct_of_total
Mkt-RF,1.0088,0.0007,0.7806
SMB,0.5392,-0.0001,-0.1574
HML,-0.1021,0.0000,0.0268
alpha,NaN,0.0003,0.3500



linear_regression(AAPL, SPY): const=0.000119, market=1.126919, R2=0.4356
rolling_regression(window=90): 912 non-NaN windows, beta range [0.23, 1.53]


## 12. Cross-Test: Construction Constraints vs PyPortfolioOpt

`WeightBounds` against `EfficientFrontier`'s `weight_bounds=` constructor
argument, and `GroupCap` against `add_sector_constraints()`.
`TurnoverCap`/`NetExposure`/`GrossExposure` have no PyPortfolioOpt equivalent
to cross-test against (PyPortfolioOpt doesn't offer a turnover-cap or
gross-exposure constraint), so they're verified by direct self-consistency
checks instead - the resulting weights are checked against the constraint's
own defining inequality/equality.

In [14]:
# --- WeightBounds vs EfficientFrontier(weight_bounds=...) ---
wb = WeightBounds(low=0.05, high=0.35)
w_wb_pp = opt.min_variance(cov, constraints=[wb])
w_wb_ref = pd.Series(EfficientFrontier(mu, cov, weight_bounds=(0.05, 0.35)).min_volatility())
wb_diff = float((w_wb_pp.reindex(symbols) - w_wb_ref.reindex(symbols)).abs().max())
print(f"WeightBounds(low=0.05, high=0.35) + min_variance vs EfficientFrontier(weight_bounds=(0.05, 0.35)): max abs diff = {wb_diff:.2e}")

# --- GroupCap vs add_sector_constraints ---
groups = {"Equity": ["AAPL", "MSFT", "JPM"], "Alt": ["TLT", "GLD"]}
gc = GroupCap(groups=groups, max_weight={"Equity": 0.6, "Alt": 0.5})
w_gc_pp = opt.min_variance(cov, constraints=[gc])

ef_gc = EfficientFrontier(mu, cov)
sector_mapper = {sym: grp for grp, members in groups.items() for sym in members}
ef_gc.add_sector_constraints(sector_mapper, {"Equity": 0.0, "Alt": 0.0}, {"Equity": 0.6, "Alt": 0.5})
w_gc_ref = pd.Series(ef_gc.min_volatility())
gc_diff = float((w_gc_pp.reindex(symbols) - w_gc_ref.reindex(symbols)).abs().max())
print(f"GroupCap({{'Equity': 0.6, 'Alt': 0.5}}) + min_variance vs add_sector_constraints: max abs diff = {gc_diff:.2e}")

# --- TurnoverCap: self-consistency (no external equivalent) ---
current = pd.Series({"AAPL": 0.30, "MSFT": 0.30, "JPM": 0.20, "TLT": 0.10, "GLD": 0.10})
tc = TurnoverCap(max_turnover=0.20, current_weights=current)
w_tc = opt.min_variance(cov, constraints=[tc])
turnover = float((w_tc.reindex(symbols) - current.reindex(symbols)).abs().sum())
print(f"\nTurnoverCap(max_turnover=0.20): realized one-way*2 turnover = {turnover:.6f} (constraint: <= 0.20)")

# --- GrossExposure + WeightBounds(allow shorts): a 130/30-style book ---
wb_short = WeightBounds(low=-0.3, high=1.0)
ge = GrossExposure(target=1.6)
w_ge = opt.max_sharpe(mu, cov, rf=0.0, constraints=[wb_short, ge])
print(f"GrossExposure(target=1.6) + WeightBounds(-0.3, 1.0) + max_sharpe: sum(abs(w)) = {float(w_ge.abs().sum()):.6f} (target: 1.6)")
print(w_ge.round(4))

# --- NetExposure: applied automatically, verify explicitly too ---
from portpy.models.base import NetExposure
ne = NetExposure(target=1.0)
w_ne = opt.min_variance(cov, constraints=[ne])
print(f"\nNetExposure(target=1.0) + min_variance: sum(w) = {float(w_ne.sum()):.10f} (target: 1.0)")

WeightBounds(low=0.05, high=0.35) + min_variance vs EfficientFrontier(weight_bounds=(0.05, 0.35)): max abs diff = 2.05e-07
GroupCap({'Equity': 0.6, 'Alt': 0.5}) + min_variance vs add_sector_constraints: max abs diff = 1.03e-09



TurnoverCap(max_turnover=0.20): realized one-way*2 turnover = 0.200000 (constraint: <= 0.20)
GrossExposure(target=1.6) + WeightBounds(-0.3, 1.0) + max_sharpe: sum(abs(w)) = 1.600000 (target: 1.6)
AAPL    0.1480
MSFT    0.1053
JPM     0.4177
TLT    -0.3000
GLD     0.6290
Name: weight, dtype: float64

NetExposure(target=1.0) + min_variance: sum(w) = 1.0000000000 (target: 1.0)


## 13. `construction.optimize`/`build`/`efficient_frontier` Recipe Layer

The dispatch layer most users actually call - already exercised indirectly
above via `portpy.models.optimization` directly, so this section confirms the
`ModelResult`-wrapping recipe functions dispatch to the exact same
numbers.

In [15]:
result_direct = opt.min_variance(cov)
result_optimize = pp_optimize(cov_matrix=cov, method="min_variance")
result_build = pp_build(y=returns, method="min_variance", covariance_kwargs={"method": "sample"})

diff_optimize = float((result_direct.reindex(symbols) - result_optimize.weights.reindex(symbols)).abs().max())
diff_build = float((result_direct.reindex(symbols) - result_build.weights.reindex(symbols)).abs().max())
print(f"optimization.min_variance vs construction.optimize(method='min_variance'): max abs diff = {diff_optimize:.2e}")
print(f"optimization.min_variance vs construction.build(method='min_variance'): max abs diff = {diff_build:.2e}")
print(f"\nModelResult diagnostics (optimize): {result_optimize.diagnostics}")

result_ef = pp_efficient_frontier(mu, cov, n_points=9)
print(f"\nconstruction.efficient_frontier: name={result_ef.name!r}, max-Sharpe weights=\n{result_ef.weights.round(4)}")

optimization.min_variance vs construction.optimize(method='min_variance'): max abs diff = 0.00e+00
optimization.min_variance vs construction.build(method='min_variance'): max abs diff = 0.00e+00

ModelResult diagnostics (optimize): {'method': 'min_variance', 'success': True, 'objective_value': 0.012202182290760458, 'iterations': 13, 'message': 'Optimization terminated successfully', 'n_restarts': 4}

construction.efficient_frontier: name='efficient_frontier', max-Sharpe weights=
AAPL   0.1035
MSFT   0.0994
JPM    0.3116
TLT    0.0339
GLD    0.4516
Name: weight, dtype: float64


## 14. Management Layer: `rebalance`, `monitor`, `compare`

No external library equivalent (this is PortPy's own post-construction
tooling), so each is checked by self-consistency instead: `rebalance`'s
trade list must actually reach the target for above-threshold assets and
leave below-threshold assets untouched; `monitor` must flag exactly the
limits its input weights actually violate; `compare` must report a turnover
consistent with the two weight vectors given.

In [16]:
current_weights = pd.Series({"AAPL": 0.30, "MSFT": 0.30, "JPM": 0.20, "TLT": 0.10, "GLD": 0.10})
target_result = pp_optimize(expected_returns=mu, cov_matrix=cov, method="max_sharpe", rf=0.0)

# --- rebalance ---
rb = rebalance(target_result, current_weights, method="threshold", threshold=0.05, cost_bps=10.0)
drift = (target_result.weights.reindex(symbols) - current_weights.reindex(symbols)).abs()
should_trade = drift > 0.05
actually_traded = rb.diagnostics["trades"].reindex(symbols).abs() > 1e-9
consistent = bool((should_trade == actually_traded).all())
print(f"rebalance(threshold=0.05): trade mask matches |target-current|>threshold exactly: {consistent}")
print(f"  turnover={rb.diagnostics['turnover']:.4f}, estimated_cost={rb.diagnostics['estimated_cost']:.6f}")

# --- monitor ---
breach_check = monitor(target_result.weights, limits={"max_weight": 0.4, "groups": groups, "max_group": {"Equity": 0.5}})
manual_max_weight_breaches = int((target_result.weights > 0.4).sum())
manual_group_breach = float(target_result.weights.reindex(groups["Equity"]).fillna(0.0).sum()) > 0.5
print(f"\nmonitor: reported {breach_check['n_breaches']} breach(es); manual max_weight count={manual_max_weight_breaches}, manual Equity-group breach={manual_group_breach}")
print(f"  breaches: {breach_check['breaches']}")

# --- compare ---
cmp = compare(current_weights, target_result)
manual_turnover = float((target_result.weights.reindex(symbols) - current_weights.reindex(symbols)).abs().sum()) / 2.0
print(f"\ncompare: reported turnover={cmp.diagnostics['turnover']:.6f} vs manually computed={manual_turnover:.6f} (mode={cmp.diagnostics['mode']!r})")

rebalance(threshold=0.05): trade mask matches |target-current|>threshold exactly: True
  turnover=0.5146, estimated_cost=0.000515

monitor: reported 2 breach(es); manual max_weight count=1, manual Equity-group breach=True
  breaches: [{'type': 'max_weight', 'asset': 'GLD', 'value': 0.4668099369688659, 'limit': 0.4}, {'type': 'max_group', 'group': 'Equity', 'value': 0.5331900630311339, 'limit': 0.5}]

compare: reported turnover=0.514613 vs manually computed=0.514613 (mode='weights_only')


## 15. Cross-Test: skfolio (Third Library)

`skfolio` solves via `cvxpy`+`CLARABEL` - a different convex-optimization stack than both
PyPortfolioOpt's `cvxpy` formulation and Riskfolio-Lib's own solver path - fitted directly
on the periodic returns (its default `EmpiricalPrior` estimates the same sample mean/covariance
PortPy's `mean_historical`/`sample` methods do). `min_variance` and `risk_parity` are
scale-invariant to the annualization convention (multiplying `Sigma` by a positive constant
doesn't move the argmin), and `max_sharpe`'s ratio is invariant too since both mu and Sigma
are scaled the same way - so skfolio's own default (periodic-scale) estimation and PortPy's
annualized `mu`/`cov` are solving the identical optimization problem underneath.

In [17]:
skfolio_results = []

def add_skfolio_res(method_name, pp_w, ref_w):
    ref_w = pd.Series(ref_w, index=returns.columns).reindex(symbols)
    pp_w = pp_w.reindex(symbols)
    max_abs_diff = float((pp_w - ref_w).abs().max())
    for sym in symbols:
        skfolio_results.append({
            "Method": method_name, "Asset": sym,
            "PortPy": pp_w[sym], "skfolio": ref_w[sym],
            "Abs Diff": abs(pp_w[sym] - ref_w[sym]), "Max Abs Diff": max_abs_diff,
        })

# --- min_variance ---
w_pp = opt.min_variance(cov)
mr_min = MeanRisk(objective_function=ObjectiveFunction.MINIMIZE_RISK, risk_measure=RiskMeasure.VARIANCE)
mr_min.fit(returns)
add_skfolio_res("min_variance", w_pp, mr_min.weights_)

# --- max_sharpe ---
w_pp = opt.max_sharpe(mu, cov, rf=0.0)
mr_sharpe = MeanRisk(objective_function=ObjectiveFunction.MAXIMIZE_RATIO, risk_measure=RiskMeasure.VARIANCE, risk_free_rate=0.0)
mr_sharpe.fit(returns)
add_skfolio_res("max_sharpe", w_pp, mr_sharpe.weights_)

# --- risk_parity ---
w_pp = opt.risk_parity(cov)
rb_sk = SkRiskBudgeting(risk_measure=RiskMeasure.VARIANCE)
rb_sk.fit(returns)
add_skfolio_res("risk_parity", w_pp, rb_sk.weights_)

df_skfolio = pd.DataFrame(skfolio_results)

def color_diff(row):
    if row["Max Abs Diff"] > 0.01:
        return ['background-color: #ffcccc'] * len(row)
    return [''] * len(row)

df_skfolio.style.apply(color_diff, axis=1).format({
    "PortPy": "{:.4f}", "skfolio": "{:.4f}", "Abs Diff": "{:.4f}", "Max Abs Diff": "{:.4f}"
})

,Method,Asset,PortPy,skfolio,Abs Diff,Max Abs Diff
0,min_variance,AAPL,0.0394,0.0391,0.0002,0.0002
1,min_variance,MSFT,0.0853,0.0854,0.0000,0.0002
2,min_variance,JPM,0.1786,0.1787,0.0001,0.0002
3,min_variance,TLT,0.4872,0.4872,0.0001,0.0002
4,min_variance,GLD,0.2095,0.2096,0.0000,0.0002
5,max_sharpe,AAPL,0.0995,0.0995,0.0000,0.0000
6,max_sharpe,MSFT,0.0859,0.0859,0.0000,0.0000
7,max_sharpe,JPM,0.3478,0.3478,0.0000,0.0000
8,max_sharpe,TLT,0.0000,0.0000,0.0000,0.0000
9,max_sharpe,GLD,0.4668,0.4668,0.0000,0.0000


## 16. Cross-Test: Negative Weights (Short Positions)

Every prior weight-based check in this notebook used the 5-asset universe from Section 3.
This section builds a separate, larger, more realistic one instead: 15 assets spanning
equities (mega-cap tech, financials, energy, staples, pharma), a long bond, a commodity, a
REIT, and two cryptocurrencies (24/7, aligned onto the equities' Mon-Fri calendar via
`core.align_calendars` - the same mixed-timeframe handling as Section 17, reused here rather
than duplicated). `max_sharpe` under a wide `WeightBounds(low=-0.3, high=0.5)` - no
`GrossExposure` this time, since neither PyPortfolioOpt nor skfolio can express a gross-
exposure cap, and adding one would silently break the cross-library comparison rather than
constrain it identically everywhere - behaves the way an aggressive, return-chasing long/short
book actually would: it funds large long positions in the highest-expected-return names by
shorting several of the weakest ones at once, not just one asset pinned at a bound. Checked
against both PyPortfolioOpt's `weight_bounds=` and skfolio's `min_weights=`/`max_weights=`,
confirming every one of the (multiple) negative weights matches across all three solver
stacks.

In [18]:
# A larger, multi-asset-class universe, built just for this section: mega-cap tech
# (chasing returns), financials, energy/staples/pharma (weaker recent momentum - plausible
# short candidates), a long bond, gold, a REIT, and two cryptocurrencies.
shorts_equity_symbols = ["AAPL", "MSFT", "NVDA", "TSLA", "AMZN", "META", "JPM", "XOM", "KO", "PFE", "TLT", "GLD", "VNQ"]
shorts_crypto_symbols = ["BTC/USD", "ETH/USD"]

shorts_equity_request = StockBarsRequest(symbol_or_symbols=shorts_equity_symbols, timeframe=TimeFrame.Day, start=start_date)
shorts_equity_bars = client.get_stock_bars(shorts_equity_request)
shorts_equity_prices = shorts_equity_bars.df["close"].unstack(level=0)
shorts_equity_prices.index = pd.to_datetime(shorts_equity_prices.index.get_level_values("timestamp") if isinstance(shorts_equity_prices.index, pd.MultiIndex) else shorts_equity_prices.index)
shorts_equity_prices = shorts_equity_prices.tz_localize(None) if shorts_equity_prices.index.tz is not None else shorts_equity_prices
shorts_equity_prices = shorts_equity_prices.dropna(how="any")

shorts_crypto_client = CryptoHistoricalDataClient()  # crypto data is public on Alpaca - no keys needed
shorts_crypto_request = CryptoBarsRequest(symbol_or_symbols=shorts_crypto_symbols, timeframe=TimeFrame.Day, start=start_date)
shorts_crypto_bars = shorts_crypto_client.get_crypto_bars(shorts_crypto_request)
shorts_crypto_prices = shorts_crypto_bars.df["close"].unstack(level=0)
shorts_crypto_prices.index = pd.to_datetime(shorts_crypto_prices.index.get_level_values("timestamp") if isinstance(shorts_crypto_prices.index, pd.MultiIndex) else shorts_crypto_prices.index)
shorts_crypto_prices = shorts_crypto_prices.tz_localize(None) if shorts_crypto_prices.index.tz is not None else shorts_crypto_prices
shorts_crypto_prices.columns = ["BTC", "ETH"]

shorts_aligned = align_calendars(pd.concat([shorts_equity_prices, shorts_crypto_prices], axis=1), method="ffill_union")
shorts_symbols = list(shorts_aligned.columns)
print(f"Extended universe: {len(shorts_symbols)} assets -> {shorts_symbols}")

shorts_weights = {s: 1.0 / len(shorts_symbols) for s in shorts_symbols}
shorts_portfolio = Portfolio(shorts_aligned, weights=shorts_weights, frequency=365, risk_free_rate=0.0)
shorts_returns = shorts_portfolio.asset_returns()
shorts_mu = pp_expected_returns(shorts_returns, method="mean_historical", periods_per_year=365)
shorts_cov = pp_covariance(shorts_returns, method="sample", periods_per_year=365)
print("\nAnnualized expected returns, sorted (the spread a return-chasing long/short book trades on):")
print(shorts_mu.sort_values().round(4))

wb_short = WeightBounds(low=-0.3, high=0.5)

w_pp_short = opt.max_sharpe(shorts_mu, shorts_cov, rf=0.0, constraints=[wb_short])
w_ref_pypfopt = pd.Series(EfficientFrontier(shorts_mu, shorts_cov, weight_bounds=(-0.3, 0.5)).max_sharpe(risk_free_rate=0.0))

mr_short = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO, risk_measure=RiskMeasure.VARIANCE,
    min_weights=-0.3, max_weights=0.5, risk_free_rate=0.0,
)
mr_short.fit(shorts_returns)
w_ref_skfolio = pd.Series(mr_short.weights_, index=shorts_returns.columns)

shorts_compare = pd.DataFrame({
    "PortPy": w_pp_short.reindex(shorts_symbols),
    "PyPortfolioOpt": w_ref_pypfopt.reindex(shorts_symbols),
    "skfolio": w_ref_skfolio.reindex(shorts_symbols),
}).sort_values("PortPy")
shorts_compare["Abs Diff vs PyPortfolioOpt"] = (shorts_compare["PortPy"] - shorts_compare["PyPortfolioOpt"]).abs()
shorts_compare["Abs Diff vs skfolio"] = (shorts_compare["PortPy"] - shorts_compare["skfolio"]).abs()

n_shorts = int((w_pp_short < -1e-6).sum())
print(f"\nmax_sharpe, 15-asset universe, WeightBounds(low=-0.3, high=0.5): {n_shorts} genuine short positions")
display(shorts_compare.round(4))

assert n_shorts >= 2, f"expected multiple short (negative) weights, got {n_shorts}"
assert shorts_compare["Abs Diff vs PyPortfolioOpt"].max() < 1e-3
assert shorts_compare["Abs Diff vs skfolio"].max() < 1e-3

Extended universe: 15 assets -> ['AAPL', 'AMZN', 'GLD', 'JPM', 'KO', 'META', 'MSFT', 'NVDA', 'PFE', 'TLT', 'TSLA', 'VNQ', 'XOM', 'BTC', 'ETH']

Annualized expected returns, sorted (the spread a return-chasing long/short book trades on):
PFE    -0.0579
TLT    -0.0361
VNQ     0.0164
KO      0.0678
XOM     0.1000
TSLA    0.1264
MSFT    0.1291
AMZN    0.1371
AAPL    0.1376
GLD     0.1496
JPM     0.1778
ETH     0.2015
BTC     0.2696
META    0.2728
NVDA    0.3503
Name: expected_return, dtype: float64

max_sharpe, 15-asset universe, WeightBounds(low=-0.3, high=0.5): 6 genuine short positions


,PortPy,PyPortfolioOpt,skfolio,Abs Diff vs PyPortfolioOpt,Abs Diff vs skfolio
VNQ,-0.3000,-0.3000,-0.3000,0.0000,0.0000
PFE,-0.2385,-0.2385,-0.2388,0.0000,0.0003
TLT,-0.1845,-0.1845,-0.1851,0.0000,0.0006
ETH,-0.0900,-0.0900,-0.0900,0.0000,0.0000
TSLA,-0.0287,-0.0287,-0.0287,0.0000,0.0000
AMZN,-0.0265,-0.0265,-0.0265,0.0000,0.0000
MSFT,0.0234,0.0234,0.0233,0.0000,0.0001
NVDA,0.0535,0.0535,0.0536,0.0000,0.0001
AAPL,0.0796,0.0796,0.0797,0.0000,0.0001
XOM,0.0975,0.0975,0.0974,0.0000,0.0001


## 17. Cross-Test: A Genuinely Mixed-Timeframe Universe

Every check above used five Mon-Fri equities on a shared calendar. Adding `BTC/USD`
(24/7, via Alpaca's public crypto endpoint - no API key needed) means the raw price frames
don't even share an index shape before `core.align_calendars` fixes that - the exact
scenario `docs/guide/calendars-and-currency.md` exists for. `min_variance` is then
cross-tested on the aligned, 6-asset, `frequency=365` universe.

In [19]:
crypto_client = CryptoHistoricalDataClient()  # crypto data is public on Alpaca - no keys needed
crypto_request = CryptoBarsRequest(symbol_or_symbols=["BTC/USD"], timeframe=TimeFrame.Day, start=start_date)
crypto_bars = crypto_client.get_crypto_bars(crypto_request)
btc_df = crypto_bars.df
btc_prices = btc_df["close"].unstack(level=0)
btc_prices.index = pd.to_datetime(btc_prices.index.get_level_values("timestamp") if isinstance(btc_prices.index, pd.MultiIndex) else btc_prices.index)
btc_prices = btc_prices.tz_localize(None) if btc_prices.index.tz is not None else btc_prices
btc_prices.columns = ["BTC"]

print(f"Equities: {len(prices)} rows, {prices.index.min().date()} -> {prices.index.max().date()}, Mon-Fri only")
print(f"BTC/USD:  {len(btc_prices)} rows, {btc_prices.index.min().date()} -> {btc_prices.index.max().date()}, 7 days/week")
print(f"Raw frames share a calendar: {prices.index.equals(btc_prices.index)}")

combined_raw = pd.concat([prices, btc_prices], axis=1)
aligned = align_calendars(combined_raw, method="ffill_union")
print(f"\nAfter align_calendars(method='ffill_union'): {aligned.shape[0]} rows, any NaN left: {aligned.isna().any().any()}")

mixed_symbols = symbols + ["BTC"]
mixed_weights = {s: 1.0 / len(mixed_symbols) for s in mixed_symbols}
mixed_portfolio = Portfolio(aligned, weights=mixed_weights, frequency=365, risk_free_rate=0.0)
mixed_returns = mixed_portfolio.asset_returns()
mixed_mu = pp_expected_returns(mixed_returns, method="mean_historical", periods_per_year=365)
mixed_cov = pp_covariance(mixed_returns, method="sample", periods_per_year=365)

w_pp_mixed = opt.min_variance(mixed_cov)
w_ref_mixed = pd.Series(EfficientFrontier(mixed_mu, mixed_cov).min_volatility())
mixed_diff = float((w_pp_mixed.reindex(mixed_symbols) - w_ref_mixed.reindex(mixed_symbols)).abs().max())

print(f"\nmin_variance on the aligned 6-asset (5 equities + BTC) universe:")
display(w_pp_mixed.round(4))
print(f"max abs diff vs PyPortfolioOpt on the same aligned data: {mixed_diff:.2e}")
assert mixed_diff < 1e-4

Equities: 1002 rows, 2022-09-15 -> 2026-09-14, Mon-Fri only
BTC/USD:  1462 rows, 2022-09-15 -> 2026-09-15, 7 days/week
Raw frames share a calendar: False

After align_calendars(method='ffill_union'): 2463 rows, any NaN left: False

min_variance on the aligned 6-asset (5 equities + BTC) universe:


AAPL   0.0371
MSFT   0.0808
JPM    0.1687
TLT    0.4634
GLD    0.1977
BTC    0.0524
Name: weight, dtype: float64

max abs diff vs PyPortfolioOpt on the same aligned data: 4.23e-09


## 18. Summary

Every estimator (4 expected-return methods, 5 covariance methods), every optimizer (all 11:
`mean_variance`, `min_variance`, `max_sharpe`, `target_return`, `target_volatility`,
`efficient_frontier`, `black_litterman`, `hierarchical_risk_parity`, `risk_parity`,
`risk_budgeting`, `maximum_diversification`), every constraint object (`WeightBounds`,
`GroupCap`, `TurnoverCap`, `NetExposure`, `GrossExposure`), the full `construction` recipe
layer, every factor-model function, and the full `management` layer
(`rebalance`/`monitor`/`compare`) were exercised above - against **three** independent
libraries (PyPortfolioOpt, Riskfolio-Lib, skfolio) or, where no external equivalent exists,
an independent manual implementation or a direct self-consistency check. Negative weights
(genuine short positions, not just unconstrained-but-unused bounds) and a calendar-aligned
mixed-timeframe universe (Mon-Fri equities plus 24/7 crypto) were both cross-tested
explicitly, not just exercised long-only/single-calendar.

Three genuine methodological differences were found and documented rather than hidden:
`shrinkage`'s target matrix (diagonal-of-sample vs scaled identity), `robust`'s MCD variant
(reweighted `MinCovDet` vs raw `fast_mcd`), and `target_return`'s equality-vs-`>=` semantics
relative to PyPortfolioOpt's `efficient_return`. Everything else matched to numerical
precision - matching the same guarantees `tests/validation/test_models_vs_libraries.py`
enforces on synthetic data as part of the automated test suite.